# Final experiment series

Consolidated notebook with three experiment series targeted at proving the
value of the trim/extend edit head. All experiments seed methods with the
**NX-heuristic init** (`build_nx_heuristic_routes` with the
`BENCHMARK_SPECS` contract). All numeric tables and figures are saved to
`artifacts/results/` with the `final_` prefix so this notebook's outputs
do not collide with the other notebooks.

* **§A. Cross-city benchmark across alpha** -- reproduces Holliday et al.
  Tables 3 (passenger perspective, `alpha = 1`) and 4 (operator
  perspective, `alpha = 0`), plus a balanced `alpha = 0.5` row, on Mandl +
  Mumford0..Mumford3. Methods compared: NSGA-II, SA, GA, HH, BCO
  (heuristic type-1 + type-2), NBCO (neural type-1 + type-2),
  Heuristic + trim/extend (type-1 + type-5 split 5+5), Extend/trim
  edit-only (all type-5), RL improvement only.

* **§B. Figure-5 ablation of the trim/extend head** -- 2x2 ablation on
  Mumford0: construction half in {type-1 trained, RPC (paper's
  pi_random)} crossed with edit half in {type-2 heuristic, trim/extend
  edit model}. Pareto figure C_p vs C_o across `alpha in [0, 1]`. The gap
  between the `type-1` and `RPC` curves at the same edit head measures
  what trained construction contributes; the gap between `type-2` and
  `trim/extend` at the same construction half measures what the trim
  head contributes.

* **§C. Trim-grace ablation on Mandl** -- for "Extend/trim edit-only BCO
  (all 10 type-5 bees)" we sweep `trim_grace_period` in `{0, 1, 3, 5,
  10}` with `worse_accept` ON, on Mandl. Quantifies the contribution of
  the trim-grace mechanism we added to `bee_colony.py`.

Edit-model weights (`EDIT_MODEL_WEIGHTS_PATH`) must exist on disk for any
method that uses `n_type5_bees > 0` (BCO Heuristic+trim/extend, Extend/
trim edit-only, the trim/extend variants in §B, and §C).

## 1. Imports + global config

In [ ]:
from pathlib import Path
import math
import gc

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import os as _os, sys as _sys
_NB_DIR = _os.path.abspath(".")
if _NB_DIR not in _sys.path:
    _sys.path.insert(0, _NB_DIR)
import eval_lib
from eval_lib import *  # noqa: F401,F403
from eval_lib import plots as _plots, _run_baseline

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

FINAL_RESULTS_SUBDIR = "final"  # all save_table calls below use subdir=FINAL_RESULTS_SUBDIR
FINAL_RESULTS_DIR = RESULTS_DIR / FINAL_RESULTS_SUBDIR
FINAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("eval_lib OK; RESULTS_DIR =", RESULTS_DIR.relative_to(ROOT_DIR))
print(f"final outputs -> artifacts/results/{FINAL_RESULTS_SUBDIR}/")
print(f"EDIT_MODEL_WEIGHTS_PATH exists: {EDIT_MODEL_WEIGHTS_PATH.exists()}")
print(f"MODEL_WEIGHTS_PATH exists:      {MODEL_WEIGHTS_PATH.exists()}")
print(f"BENCHMARK_SPECS ({len(BENCHMARK_SPECS)} cities) and BCO_VARIANTS "
      f"({len(BCO_VARIANTS)} entries) in scope")

## 2. Section A: Cross-city benchmark across alpha

For every `(city, alpha)` pair we run every method from the same NX-
heuristic init under cost weights
`(demand_time_weight = alpha, route_time_weight = 1 - alpha,
median_connectivity_weight = 0)`. This sweeps the operator -> balanced ->
passenger trade-off the paper isolates as Tables 3 and 4. After the
sweep three display tables are built -- one per alpha value -- with
columns matching Holliday's: City, Method, C_p (ATT), C_o (RTT),
d_0, d_1, d_2, d_un, cost.

NSGA-II is multi-objective: its `_run_baseline` post-eval uses the same
alpha-weighted cost as everyone else, but its Pareto front is reduced
to the lowest-weighted-sum member via `reduce_pareto_front(output,
demand_weight=alpha, route_weight=1-alpha)`.

SA / GA / HH cost-function weights are hardcoded in
`_baseline_cfg_overrides` from `eval_lib.params.DEMAND/ROUTE/CONN_WEIGHT`;
we override them on the composed cfg via direct attribute assignment
before running.

In [ ]:
ALPHA_GRID = [0.0, 0.5, 1.0]
ALPHA_LABELS = {0.0: "operator (alpha=0)",
                0.5: "balanced (alpha=0.5)",
                1.0: "passenger (alpha=1)"}

# Methods to compare. Names match the user spec exactly.
A_METHODS = [
    ("NSGA-II",                                    "nsgaii"),
    ("Simulated annealing",                        "sa"),
    ("Genetic algorithm",                          "ga"),
    ("Hyper-heuristics",                           "hh"),
    ("BCO",                                        "bco_variant:seeded_heuristic"),
    ("NBCO",                                       "bco_variant:seeded_neural"),
    ("Heuristic BCO Type 1 + trim/extend",         "bco_variant:heuristic_extend_trim_split_5_5"),
    ("BCO: Extend/trim edit-only BCO (all 10 bees)", "bco_variant:extend_trim_edit_only"),
    ("RL improvement only",                        "rl_only"),
    ("NX-heuristic (initial)",                     "initial"),
]

BCO_VARIANT_LOOKUP = {v["key"]: v for v in BCO_VARIANTS}

A_CITIES = ["Mandl", "Mumford0", "Mumford1", "Mumford2", "Mumford3"]
A_SEED = 0  # single seed sweep -- bump if you need confidence intervals
REUSE_EXISTING_A_SWEEP = True

print(f"Section A matrix: {len(A_CITIES)} cities x {len(ALPHA_GRID)} alphas "
      f"x {len(A_METHODS)} methods = "
      f"{len(A_CITIES) * len(ALPHA_GRID) * len(A_METHODS)} runs")

In [ ]:
def _set_cfg_weights(cfg, alpha):
    """Override (demand, route, conn) cost weights in a composed cfg
    in-place. Used for SA/GA/HH/NSGA-II where the builder hardcodes
    DEMAND/ROUTE/CONN_WEIGHT defaults from eval_lib.params."""
    cfg.experiment.cost_function.kwargs.demand_time_weight = float(alpha)
    cfg.experiment.cost_function.kwargs.route_time_weight = float(1.0 - alpha)
    cfg.experiment.cost_function.kwargs.median_connectivity_weight = 0.0
    return cfg


def _alpha_weights_dict(alpha):
    return {"demand_time_weight": float(alpha),
            "route_time_weight": float(1.0 - alpha),
            "median_connectivity_weight": 0.0}


def _run_one(city, alpha, method_label, method_kind, spec, tensors, init_routes):
    """Run one (city, alpha, method) combo; return a flat row dict."""
    weights = _alpha_weights_dict(alpha)
    nr, mn, mx = spec["n_routes"], spec["min_route_len"], spec["max_route_len"]
    run_scope = f"final_a_{city}_a{alpha:.1f}_"

    if method_kind == "initial":
        cfg = build_sa_cfg(f"{run_scope}initial", nr, mn, mx)
        _set_cfg_weights(cfg, alpha)
        _, metrics, _, routes = _run_baseline(
            None, cfg, init_routes, f"{run_scope}initial_", {}, tensors=tensors)
    elif method_kind == "sa":
        cfg = build_sa_cfg(f"{run_scope}sa", nr, mn, mx)
        _set_cfg_weights(cfg, alpha)
        _, metrics, _, routes = run_sa(cfg, init_routes, tensors=tensors,
                                        run_name_scope=run_scope)
    elif method_kind == "ga":
        cfg = build_ga_cfg(f"{run_scope}ga", nr, mn, mx)
        _set_cfg_weights(cfg, alpha)
        _, metrics, _, routes = run_ga(cfg, init_routes, tensors=tensors,
                                        run_name_scope=run_scope)
    elif method_kind == "hh":
        cfg = build_hh_cfg(f"{run_scope}hh", nr, mn, mx)
        _set_cfg_weights(cfg, alpha)
        _, metrics, _, routes = run_hh(cfg, init_routes, tensors=tensors,
                                        run_name_scope=run_scope)
    elif method_kind == "nsgaii":
        cfg = build_nsgaii_cfg(f"{run_scope}nsgaii", nr, mn, mx)
        _set_cfg_weights(cfg, alpha)
        _, output = run_nsgaii(cfg, tensors=tensors, init_routes=init_routes,
                                run_name_scope=run_scope)
        best = reduce_pareto_front(output, weights["demand_time_weight"],
                                    weights["route_time_weight"])
        picked = best["routes"]
        if picked.ndim == 2: picked = picked[None]
        eval_cfg = build_sa_cfg(f"{run_scope}nsgaii_eval", nr, mn, mx)
        _set_cfg_weights(eval_cfg, alpha)
        _, metrics, _, routes = _run_baseline(
            None, eval_cfg, picked, f"{run_scope}nsgaii_eval_", {},
            tensors=tensors)
    elif method_kind == "rl_only":
        _, metrics, _, routes, *_rest = run_rl_improvement(
            init_routes, run_name=f"{run_scope}rl_only",
            n_routes=nr, min_route_len=mn, max_route_len=mx,
            tensors=tensors, weights=weights)
    elif method_kind.startswith("bco_variant:"):
        key = method_kind.split(":", 1)[1]
        variant = BCO_VARIANT_LOOKUP[key]
        cfg = build_bco_cfg(
            run_name=f"{run_scope}{variant['run_name']}",
            n_routes=nr, min_route_len=mn, max_route_len=mx,
            use_neural_bees=variant["use_neural_bees"],
            n_type1_bees=variant["n_type1_bees"],
            n_type2_bees=variant["n_type2_bees"],
            n_type4_bees=variant["n_type4_bees"],
            n_type5_bees=variant.get("n_type5_bees", 0),
            n_type6_bees=variant.get("n_type6_bees", 0),
            n_type7_bees=variant.get("n_type7_bees", 0),
            **weights)
        cfg.experiment.seed = A_SEED
        _, metrics, _, routes, _mc = run_bco(
            cfg, init_routes, tensors=tensors, run_name_scope=run_scope)
    else:
        raise ValueError(f"Unknown method kind: {method_kind}")

    row = {"city": city, "alpha": alpha, "method": method_label,
           "ATT": float(metric_value(metrics, "ATT")),
           "RTT": float(metric_value(metrics, "RTT")),
           "d_0": float(metric_value(metrics, "$d_0$")),
           "d_1": float(metric_value(metrics, "$d_1$")),
           "d_2": float(metric_value(metrics, "$d_2$")),
           "d_un": float(metric_value(metrics, "$d_{un}$")),
           "cost": float(metric_value(metrics, "cost"))}
    return row, routes

In [ ]:
if REUSE_EXISTING_A_SWEEP and "final_a_rows" in globals():
    print("Reusing existing final_a_rows; "
          "set REUSE_EXISTING_A_SWEEP=False to rerun.")
else:
    final_a_rows = []
    city_to_spec_a = {s["city"]: s for s in BENCHMARK_SPECS}
    for city in A_CITIES:
        spec = city_to_spec_a[city]
        tensors, init_routes = load_benchmark_graph(spec)
        for alpha in ALPHA_GRID:
            for label, kind in A_METHODS:
                print(f"  [{city} | alpha={alpha} | {label}] running...")
                try:
                    row, _routes = _run_one(
                        city, alpha, label, kind, spec, tensors, init_routes)
                    final_a_rows.append(row)
                    print(f"    -> cost={row['cost']:.4f}, ATT={row['ATT']:.2f}, "
                          f"RTT={row['RTT']:.2f}")
                except Exception as exc:
                    print(f"    FAILED: {exc!r}")
                    final_a_rows.append({"city": city, "alpha": alpha,
                                         "method": label,
                                         "ATT": float("nan"), "RTT": float("nan"),
                                         "d_0": float("nan"), "d_1": float("nan"),
                                         "d_2": float("nan"), "d_un": float("nan"),
                                         "cost": float("nan")})
                finally:
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

final_a_df = pd.DataFrame(final_a_rows)
save_table(final_a_df, "benchmark_alpha_rows", subdir=FINAL_RESULTS_SUBDIR)
print(f"\n{len(final_a_df)} rows total")
display(final_a_df.head(10).round(3))

### 3. Per-alpha display tables (Holliday Tables 3 + 4 shape)

For each `alpha` value we pivot the rows into a `City x Method` table
with the headline metrics (`C_p`, `C_o`, `d_0`, `d_1`, `d_2`, `d_un`).
Saved per-alpha as
`artifacts/results/final_benchmark_alpha_<alpha>.csv`.

In [ ]:
METRIC_COLS_ALPHA = ["ATT", "RTT", "d_0", "d_1", "d_2", "d_un", "cost"]

for alpha in ALPHA_GRID:
    alpha_df = final_a_df[final_a_df["alpha"] == alpha].copy()
    if alpha_df.empty:
        continue
    print(f"\n========== alpha = {alpha} ({ALPHA_LABELS[alpha]}) ==========")
    show_cols = ["city", "method"] + METRIC_COLS_ALPHA
    out = alpha_df[show_cols].reset_index(drop=True).round(3)
    save_table(out, f"benchmark_alpha_{alpha:.1f}", subdir=FINAL_RESULTS_SUBDIR)
    display(out)

## 4. Section B: Figure-5 ablation of trim/extend head (Mumford0)

2x2 ablation -- construction half in {type-1 trained, RPC} crossed with
edit half in {type-2 heuristic, trim/extend trained edit model}. Both
halves contribute 5 bees out of 10. RPC = paper's pi_random (random
shortest-path combiner; no trained weights). The notebook-local variants
live in `B_ABLATION_VARIANTS` below; they do not leak into the library
`BCO_VARIANTS`.

Pareto figure: x = ATT (C_p), y = RTT (C_o), one curve per variant,
points connected in `alpha` order. The gap between the (type-1, ·) and
(RPC, ·) curves at the same edit head measures trained construction's
contribution; the gap between (·, type-2) and (·, trim/extend) measures
the trim head's contribution.

In [ ]:
# Notebook-local ablation variants -- intentionally NOT in library BCO_VARIANTS
# so the other experiment sections do not see them.
B_ABLATION_VARIANTS = [
    {  # type1 + type2 = paper's EA analog (classical BCO)
        "key": "type1_type2",
        "summary_label": "type1 + type2 (EA analog)",
        "run_name": "final_b_type1_type2",
        "use_neural_bees": False,
        "n_type1_bees": 5, "n_type2_bees": 5,
        "n_type4_bees": 0, "n_type5_bees": 0, "n_type6_bees": 0, "n_type7_bees": 0,
    },
    {  # RPC + type2 = paper's RC-EA analog
        "key": "rpc_type2",
        "summary_label": "RPC + type2 (RC-EA analog)",
        "run_name": "final_b_rpc_type2",
        "use_neural_bees": False,
        "n_type1_bees": 0, "n_type2_bees": 5,  # n_type3 = 10 - 5 = 5 (auto)
        "n_type4_bees": 0, "n_type5_bees": 0, "n_type6_bees": 0, "n_type7_bees": 0,
    },
    {  # type1 + trim/extend = our smart variant on classical SP rebuild
        "key": "type1_trim_extend",
        "summary_label": "type1 + trim/extend",
        "run_name": "final_b_type1_trim_extend",
        "use_neural_bees": False,
        "n_type1_bees": 5, "n_type2_bees": 0,
        "n_type4_bees": 0, "n_type5_bees": 5, "n_type6_bees": 0, "n_type7_bees": 0,
    },
    {  # RPC + trim/extend = our smart variant on random-path construction
        "key": "rpc_trim_extend",
        "summary_label": "RPC + trim/extend",
        "run_name": "final_b_rpc_trim_extend",
        "use_neural_bees": False,
        "n_type1_bees": 0, "n_type2_bees": 0,  # n_type3 = 10 - 5 = 5 (auto)
        "n_type4_bees": 0, "n_type5_bees": 5, "n_type6_bees": 0, "n_type7_bees": 0,
    },
]
print(f"§B variants ({len(B_ABLATION_VARIANTS)}):")
for v in B_ABLATION_VARIANTS:
    print(f"  {v['summary_label']}")

B_CITY = "Mumford0"
B_ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]
B_SEED = 0
B_ACCEPT_MODE = "without_worse"
REUSE_EXISTING_B_SWEEP = True
print(f"\n§B sweep: city={B_CITY}, alphas={B_ALPHAS}, seed={B_SEED}, "
      f"accept={B_ACCEPT_MODE}")

In [ ]:
if REUSE_EXISTING_B_SWEEP and "final_b_rows" in globals():
    print("Reusing existing final_b_rows; "
          "set REUSE_EXISTING_B_SWEEP=False to rerun.")
else:
    final_b_rows = []
    city_to_spec_b = {s["city"]: s for s in BENCHMARK_SPECS}
    spec_b = city_to_spec_b[B_CITY]
    tensors_b, init_routes_b = load_benchmark_graph(spec_b)

    for alpha in B_ALPHAS:
        weights = _alpha_weights_dict(alpha)
        for variant in B_ABLATION_VARIANTS:
            label = variant["summary_label"]
            print(f"  [§B alpha={alpha} | {label}] running...")
            try:
                cfg = build_bco_cfg(
                    run_name=f"final_b_a{alpha:.2f}_{variant['run_name']}",
                    n_routes=spec_b["n_routes"],
                    min_route_len=spec_b["min_route_len"],
                    max_route_len=spec_b["max_route_len"],
                    use_neural_bees=variant["use_neural_bees"],
                    n_type1_bees=variant["n_type1_bees"],
                    n_type2_bees=variant["n_type2_bees"],
                    n_type4_bees=variant["n_type4_bees"],
                    n_type5_bees=variant.get("n_type5_bees", 0),
                    n_type6_bees=variant.get("n_type6_bees", 0),
                    n_type7_bees=variant.get("n_type7_bees", 0),
                    **weights)
                cfg.experiment.seed = B_SEED
                _, metrics, _, _routes, _mc = run_bco(
                    cfg, init_routes_b, tensors=tensors_b,
                    run_name_scope=f"final_b_a{alpha:.2f}_")
                row = {"alpha": alpha, "method": label,
                       "ATT": float(metric_value(metrics, "ATT")),
                       "RTT": float(metric_value(metrics, "RTT")),
                       "cost": float(metric_value(metrics, "cost")),
                       "d_un": float(metric_value(metrics, "$d_{un}$"))}
                final_b_rows.append(row)
                print(f"    -> cost={row['cost']:.4f}, ATT={row['ATT']:.2f}, "
                      f"RTT={row['RTT']:.2f}")
            except Exception as exc:
                print(f"    FAILED: {exc!r}")
            finally:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

final_b_df = pd.DataFrame(final_b_rows)
save_table(final_b_df, "b_pareto_ablation", subdir=FINAL_RESULTS_SUBDIR)
display(final_b_df.round(3))

### Figure-5 Pareto figure (4 curves)

The 4 ablation variants on one (`ATT`, `RTT`) plot. Down-and-left = better.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
cmap = plt.get_cmap("tab10")
method_colors = {}
for i, variant in enumerate(B_ABLATION_VARIANTS):
    label = variant["summary_label"]
    method_colors[label] = cmap(i % 10)
    mdf = final_b_df[final_b_df["method"] == label]
    if mdf.empty:
        continue
    agg = mdf.groupby("alpha")[["ATT", "RTT"]].mean().reset_index().sort_values("alpha")
    ax.plot(agg["ATT"], agg["RTT"], marker="o", linestyle="-",
             linewidth=1.8, markersize=7, label=label,
             color=method_colors[label], alpha=0.9)
    # Tag the endpoints with the alpha values
    for _, r in agg.iterrows():
        ax.annotate(f"α={r['alpha']}", (r["ATT"], r["RTT"]),
                     xytext=(4, 4), textcoords="offset points",
                     fontsize=7, color=method_colors[label], alpha=0.8)
ax.set_xlabel("C_p / ATT (minutes; lower = better passenger time)")
ax.set_ylabel("C_o / RTT (minutes; lower = better operator time)")
ax.set_title(f"§B Figure-5 ablation on {B_CITY}: trim/extend head contribution",
              fontweight="bold")
ax.grid(alpha=0.25)
ax.legend(loc="best", framealpha=0.9)
fig.tight_layout()
plt.show()

## 5. Section C: Trim-grace ablation on Mandl

For "Extend/trim edit-only BCO (all 10 type-5 bees)" on Mandl we sweep
`trim_grace_period in {0, 1, 3, 5, 10}` with `worse_accept` ON. The
trim-grace mechanism (`bee_colony.py`) force-accepts a route-shrinking
mutation as a setup move and shields the bee from cost-based selection
for `trim_grace_period` rounds so a follow-up extend can build on the
trim. Larger values give trims more room to pay off; zero disables the
mechanism.

Plot: cost vs `trim_grace_period`. We expect a U-curve or monotone-down
trend up to some optimum.

In [ ]:
C_CITY = "Mandl"
C_TRIM_GRACE_VALUES = [0, 1, 3, 5, 10]
C_SEED = 0
REUSE_EXISTING_C_SWEEP = True

# We'll target the extend_trim_edit_only variant (all 10 type-5 bees).
C_VARIANT = next(v for v in BCO_VARIANTS if v["key"] == "extend_trim_edit_only")
print(f"§C variant: {C_VARIANT['summary_label']}")
print(f"§C city: {C_CITY}, trim_grace_values: {C_TRIM_GRACE_VALUES}, "
      f"seed={C_SEED}")

In [ ]:
if REUSE_EXISTING_C_SWEEP and "final_c_rows" in globals():
    print("Reusing existing final_c_rows; "
          "set REUSE_EXISTING_C_SWEEP=False to rerun.")
else:
    final_c_rows = []
    city_to_spec_c = {s["city"]: s for s in BENCHMARK_SPECS}
    spec_c = city_to_spec_c[C_CITY]
    tensors_c, init_routes_c = load_benchmark_graph(spec_c)

    for tg in C_TRIM_GRACE_VALUES:
        print(f"  [§C trim_grace_period={tg}] running...")
        try:
            # Build cfg directly to override trim_grace_period -- this is the
            # one knob run_method() doesn't expose explicitly. We also keep
            # worse_accept ON (T0=WORSE_ACCEPT_ON_TEMPERATURE) so the trim-
            # grace mechanism actually engages.
            cfg = build_bco_cfg(
                run_name=f"final_c_tg{tg}_{C_VARIANT['run_name']}",
                n_routes=spec_c["n_routes"],
                min_route_len=spec_c["min_route_len"],
                max_route_len=spec_c["max_route_len"],
                use_neural_bees=C_VARIANT["use_neural_bees"],
                n_type1_bees=C_VARIANT["n_type1_bees"],
                n_type2_bees=C_VARIANT["n_type2_bees"],
                n_type4_bees=C_VARIANT["n_type4_bees"],
                n_type5_bees=C_VARIANT.get("n_type5_bees", 0),
                n_type6_bees=C_VARIANT.get("n_type6_bees", 0),
                n_type7_bees=C_VARIANT.get("n_type7_bees", 0),
                worse_accept_temperature=WORSE_ACCEPT_ON_TEMPERATURE,
                worse_selection_temperature=WORSE_SELECTION_ON_TEMPERATURE,
                trim_grace_period=int(tg),
            )
            cfg.experiment.seed = C_SEED
            _, metrics, _, _routes, _mc = run_bco(
                cfg, init_routes_c, tensors=tensors_c,
                run_name_scope=f"final_c_tg{tg}_")
            row = {"trim_grace_period": int(tg),
                   "cost": float(metric_value(metrics, "cost")),
                   "ATT": float(metric_value(metrics, "ATT")),
                   "RTT": float(metric_value(metrics, "RTT")),
                   "d_un": float(metric_value(metrics, "$d_{un}$"))}
            final_c_rows.append(row)
            print(f"    -> cost={row['cost']:.4f}, ATT={row['ATT']:.2f}, "
                  f"RTT={row['RTT']:.2f}, d_un={row['d_un']:.2f}")
        except Exception as exc:
            print(f"    FAILED: {exc!r}")
            final_c_rows.append({"trim_grace_period": int(tg),
                                  "cost": float("nan"),
                                  "ATT": float("nan"), "RTT": float("nan"),
                                  "d_un": float("nan")})
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

final_c_df = pd.DataFrame(final_c_rows)
save_table(final_c_df, "c_trim_grace_ablation", subdir=FINAL_RESULTS_SUBDIR)
display(final_c_df.round(3))

### Trim-grace cost vs `trim_grace_period`

Cost on the y-axis (lower = better) against trim-grace value. A dip
suggests the mechanism helps; flat means no signal at this seed/budget.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(final_c_df["trim_grace_period"], final_c_df["cost"],
         marker="o", linewidth=1.8, markersize=7, color="tab:blue",
         label="cost")
ax.set_xlabel("trim_grace_period")
ax.set_ylabel("cost  (lower = better)")
ax.set_title(f"§C trim-grace ablation on {C_CITY} "
              f"(Extend/trim edit-only, worse_accept ON)",
              fontweight="bold")
ax.grid(alpha=0.25)
ax.legend(loc="best")
fig.tight_layout()
plt.show()

## 6. Outputs summary

After Run-All the following land in `artifacts/results/final/`
(a dedicated subfolder so this notebook's outputs do not mix with the
other experiment notebooks' tables):

* `benchmark_alpha_rows.csv`           -- §A raw rows (city x alpha x method).
* `benchmark_alpha_{0.0,0.5,1.0}.csv`  -- §A per-alpha display tables (one
                                          per alpha, matching Holliday's
                                          Tables 3 / 4 shape).
* `b_pareto_ablation.csv`              -- §B 4-curve Pareto rows.
* `c_trim_grace_ablation.csv`          -- §C trim-grace sweep.

Figures are shown inline only -- regenerable from the CSVs after a
kernel restart by loading the dataframes with `pd.read_csv` and
re-running the figure cells in isolation.